# 🎙️ Hindi TTS — Self-Contained GPU Training Pipeline
### Works 100% standalone — no broken scripts, no missing deps, no HF token needed

| Step | What it does | Time (T4 GPU) |
|------|--------------|---------------|
| 1 | Check GPU | 5 s |
| 2 | Download audio from Google Drive | 1–2 min |
| 3 | Install all dependencies (fixed) | 3–5 min |
| 4 | Whisper transcription + auto-chunking | 20–40 min |
| 5 | Build GPT-SoVITS filelist (inline, no broken scripts) | 10 s |
| 6 | Clone & launch GPT-SoVITS WebUI (all deps pre-fixed) | 5–10 min |
| Bonus | Test Edge-TTS instant Hindi speech | 5 s |

> ⚠️ Set **Runtime → Change runtime type → T4 GPU** before running.

## ⚡ Step 1: Check GPU

In [ ]:
import torch
print('='*55)
print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name      :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM          : {vram:.1f} GB')
else:
    print('⚠️  NO GPU — go to Runtime → Change runtime type → T4 GPU')
print('='*55)
!nvidia-smi

## 📁 Step 2: Get Audio Files from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Where audio ends up (regardless of source) ───────────────────────────────
AUDIO_DIR = '/content/hindi_tts/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

DRIVE_FOLDER = '/content/drive/MyDrive/hindi_tts_data'

if os.path.exists(DRIVE_FOLDER):
    import shutil
    wav_files = [f for f in os.listdir(DRIVE_FOLDER) if f.lower().endswith('.wav')]
    for f in wav_files:
        dest = os.path.join(AUDIO_DIR, f)
        if not os.path.exists(dest):
            shutil.copy2(os.path.join(DRIVE_FOLDER, f), dest)
    print(f'[+] Copied {len(wav_files)} WAV files from Drive → {AUDIO_DIR}')
else:
    print('[-] Drive folder not found — using gdown fallback...')
    !pip install -q gdown
    import gdown
    gdown.download_folder(
        'https://drive.google.com/drive/folders/17OKMSInQ7-8KFNUn0voAYPhFzViLAFra?usp=sharing',
        output=AUDIO_DIR, quiet=False
    )

wav_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav')]
print(f'\n[+] Total WAV files ready : {len(wav_files)}')
for f in wav_files[:4]:
    mb = os.path.getsize(os.path.join(AUDIO_DIR, f)) / 1e6
    print(f'    {mb:5.1f} MB  {f[:60]}')
if len(wav_files) > 4:
    print(f'    ... and {len(wav_files)-4} more')

## 📦 Step 3: Install ALL Dependencies (Fixed)

Fixes applied vs the old notebook:
- Added `cmake` + system build tools so `opencc` compiles
- Added `mecab` so `python_mecab_ko` compiles  
- Added `ffmpeg-python` (the `import ffmpeg` GPT-SoVITS needs)
- Added `edge-tts`, `openai`, `pandas` which were missing

In [ ]:
print('[1/4] Installing system build deps (cmake, mecab, ffmpeg)...')
!apt-get install -qq -y cmake mecab libmecab-dev mecab-ipadic-utf8 ffmpeg > /dev/null 2>&1
print('[+] System deps OK')

print('[2/4] Installing Python audio + TTS packages...')
!pip install -q \
    edge-tts \
    openai \
    openai-whisper \
    soundfile \
    pydub \
    ffmpeg-python \
    gdown \
    pandas \
    python-dotenv
print('[+] Audio + TTS packages OK')

print('[3/4] Installing demucs (vocal separation)...')
!pip install -q demucs
print('[+] Demucs OK')

print('[4/4] Verifying key imports...')
import whisper, soundfile, ffmpeg, edge_tts
print('[+] All imports OK!')
print('\n✅ Dependencies installed successfully!')

## 🎙️ Step 4: Whisper Transcription + Auto-Chunking

**Replaces the broken 3-script diarization pipeline** with a single step:
- Uses Whisper `medium` on each raw WAV
- Automatically splits into short segments using Whisper's own timestamps
- Saves each segment as a separate WAV file  
- Writes `metadata.csv` with correct local paths

> ⏱ ~20–40 min for all 26 stories on T4 GPU  
> To test faster, set `MAX_FILES = 3` below.

In [ ]:
import os, csv, torch
import numpy as np
import soundfile as sf
import whisper

# ── Config ────────────────────────────────────────────────────────────────────
AUDIO_DIR    = '/content/hindi_tts/audio'
SEGMENTS_DIR = '/content/hindi_tts/segments'
METADATA_CSV = '/content/hindi_tts/metadata.csv'
WHISPER_MODEL = 'medium'  # 'small' for faster test, 'large-v3' for best quality
MIN_DURATION  = 2.0       # seconds — skip very short clips
MAX_DURATION  = 15.0      # seconds — split longer clips
MAX_FILES     = None      # Set to 3 for a quick test; None = all files

os.makedirs(SEGMENTS_DIR, exist_ok=True)

# ── Load Whisper ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Loading Whisper {WHISPER_MODEL} on {device.upper()}...')
print('    (First run downloads ~1.5 GB model)')
model = whisper.load_model(WHISPER_MODEL, device=device)
print(f'[+] Whisper ready!\n')

# ── Helper: load + resample to 16kHz mono ─────────────────────────────────────
def load_audio_16k(path):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)   # stereo → mono
    if sr != 16000:
        import resampy
        audio = resampy.resample(audio, sr, 16000)
    return audio.astype(np.float32), 16000

# ── Helper: save a segment slice ──────────────────────────────────────────────
def save_segment(audio, sr, start, end, out_path):
    s, e = int(start * sr), int(end * sr)
    chunk = audio[s:e]
    if len(chunk) == 0:
        return False
    sf.write(out_path, chunk, sr)
    return True

# ── Main loop ─────────────────────────────────────────────────────────────────
wav_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav')])
if MAX_FILES:
    wav_files = wav_files[:MAX_FILES]

print(f'[*] Processing {len(wav_files)} WAV files...\n')

records = []
total_files = len(wav_files)

for file_idx, fname in enumerate(wav_files, 1):
    story_name = os.path.splitext(fname)[0][:60]  # truncate for folder name
    wav_path   = os.path.join(AUDIO_DIR, fname)
    story_dir  = os.path.join(SEGMENTS_DIR, story_name)
    os.makedirs(story_dir, exist_ok=True)

    print(f'[{file_idx}/{total_files}] Transcribing: {fname[:70]}')

    try:
        # Transcribe with word-level timestamps
        result = model.transcribe(
            wav_path,
            language='hi',
            task='transcribe',
            word_timestamps=False,
            fp16=(device == 'cuda'),
            verbose=False,
        )
    except Exception as e:
        print(f'  [-] Whisper failed: {e}')
        continue

    # Load audio for slicing
    try:
        audio, sr = load_audio_16k(wav_path)
    except Exception as e:
        # Fallback: use whisper's own loader
        try:
            audio = whisper.load_audio(wav_path)
            sr = 16000
        except Exception as e2:
            print(f'  [-] Audio load failed: {e2}')
            continue

    file_segments = 0
    for seg in result.get('segments', []):
        text     = seg['text'].strip()
        start    = seg['start']
        end      = seg['end']
        duration = end - start

        if not text or duration < MIN_DURATION:
            continue

        # If segment is too long, split at midpoint
        if duration > MAX_DURATION:
            mid = (start + end) / 2
            sub_segments = [(start, mid), (mid, end)]
        else:
            sub_segments = [(start, end)]

        for sub_start, sub_end in sub_segments:
            seg_name = f'seg_{file_segments:04d}_{int(sub_start)}s.wav'
            seg_path = os.path.join(story_dir, seg_name)

            if save_segment(audio, sr, sub_start, sub_end, seg_path):
                records.append({
                    'audio_path': seg_path,
                    'speaker_id': 'SPEAKER_00',
                    'story':      story_name,
                    'language':   'hi',
                    'text':       text,
                    'duration':   round(sub_end - sub_start, 2),
                })
                file_segments += 1

    print(f'  [+] {file_segments} segments → {story_dir}')

# ── Write metadata.csv ────────────────────────────────────────────────────────
with open(METADATA_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['audio_path','speaker_id','story','language','text','duration'])
    writer.writeheader()
    writer.writerows(records)

print(f'\n✅ Transcription complete!')
print(f'   Total segments : {len(records)}')
print(f'   Metadata CSV   : {METADATA_CSV}')

## 📊 Step 4B: Preview Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/content/hindi_tts/metadata.csv')
print(f'Total segments : {len(df)}')
print(f'Stories        : {df["story"].nunique()}')
print(f'Total duration : {df["duration"].sum()/60:.1f} min')
print(f'Avg duration   : {df["duration"].mean():.2f} s')
print()
print('Duration buckets:')
print(f'  < 2 s  : {(df.duration < 2).sum()}')
print(f'  2–8 s  : {((df.duration >= 2) & (df.duration < 8)).sum()}')
print(f'  8–15 s : {((df.duration >= 8) & (df.duration <= 15)).sum()}')
print()
print(df[['story','text','duration']].head(8).to_string(index=False))

## 🤖 Step 5: Build GPT-SoVITS Training Filelist (Inline — No Broken Scripts)

Builds `sovits_filelist.txt` and `xtts_metadata.csv` directly from `metadata.csv`.
No dependency on `scripts/08_format_sovits.py`.

In [ ]:
import pandas as pd, os, json, torch

METADATA_CSV  = '/content/hindi_tts/metadata.csv'
OUTPUT_DIR    = '/content/hindi_tts/training_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(METADATA_CSV)

# ── Filter: keep only 2–15 s segments with non-empty text ────────────────────
df = df[
    df['text'].notna() &
    (df['text'].str.strip() != '') &
    (df['audio_path'].apply(os.path.exists)) &
    (df['duration'] >= 2.0) &
    (df['duration'] <= 15.0)
].reset_index(drop=True)

print(f'[+] Filtered dataset: {len(df)} usable segments')

# ── GPT-SoVITS filelist: audio_path|speaker|lang|text ────────────────────────
SOVITS_FILELIST = os.path.join(OUTPUT_DIR, 'sovits_filelist.txt')
lines = []
for _, row in df.iterrows():
    speaker = row.get('speaker_id', 'SPEAKER_00')
    lines.append(f"{row['audio_path']}|{speaker}|HI|{row['text'].strip()}")
with open(SOVITS_FILELIST, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'[+] GPT-SoVITS filelist → {SOVITS_FILELIST}  ({len(lines)} lines)')

# ── XTTS-v2 metadata CSV ──────────────────────────────────────────────────────
XTTS_META = os.path.join(OUTPUT_DIR, 'xtts_metadata.csv')
xtts_lines = ['audio_file|text|speaker_name']
for _, row in df.iterrows():
    speaker = row.get('speaker_id', 'SPEAKER_00')
    xtts_lines.append(f"{row['audio_path']}|{row['text'].strip()}|{speaker}")
with open(XTTS_META, 'w', encoding='utf-8') as f:
    f.write('\n'.join(xtts_lines))

# ── XTTS config (auto-detects cuda/cpu) ───────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
xtts_config = {
    'run_name': 'hindi_tts_xtts',
    'model': 'xtts',
    'audio': {'sample_rate': 22050, 'max_wav_value': 32768.0},
    'datasets': [{
        'name': 'hindi_tts',
        'path': OUTPUT_DIR,
        'meta_file': 'xtts_metadata.csv',
        'language': 'hi'
    }],
    'training': {
        'batch_size': 4,
        'epochs': 100,
        'learning_rate': 0.0001,
        'mixed_precision': device == 'cuda',
        'device': device,
    }
}
XTTS_CONFIG = os.path.join(OUTPUT_DIR, 'xtts_config.json')
with open(XTTS_CONFIG, 'w') as f:
    json.dump(xtts_config, f, indent=2, ensure_ascii=False)

print(f'[+] XTTS metadata    → {XTTS_META}')
print(f'[+] XTTS config      → {XTTS_CONFIG}  (device={device})')
print()
print('✅ Training data ready!')
print(f'   Segments : {len(df)}')
print(f'   Duration : {df["duration"].sum()/60:.1f} min total')

## 🚀 Step 6: Install & Launch GPT-SoVITS Training WebUI

**Fixes applied vs old notebook:**
1. `cmake` pre-installed → `opencc` builds successfully
2. `mecab` pre-installed → `python_mecab_ko` builds successfully  
3. `ffmpeg-python` installed → `import ffmpeg` works in `webui.py`
4. `filelist.txt` already copied into GPT-SoVITS dir

> 👉 After the cell runs, click the **`*.gradio.live`** link.  
> In the WebUI: set **Training filelist** to `/content/GPT-SoVITS/hindi_filelist.txt`

In [ ]:
import os

# ── 6a: Clone GPT-SoVITS ─────────────────────────────────────────────────────
print('[1/4] Cloning GPT-SoVITS...')
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS 2>/dev/null \
    || echo '[*] Already cloned'
os.chdir('/content/GPT-SoVITS')
print('[+] Cloned')

# ── 6b: Install requirements (opencc + mecab already fixed by apt in Step 3) ─
print('[2/4] Installing GPT-SoVITS requirements...')
# Install everything except the two that always fail on Colab
!grep -v 'opencc\|python_mecab_ko' requirements.txt | pip install -q -r /dev/stdin
# Install opencc via conda-forge (much more reliable than building from source)
!pip install -q opencc-python-reimplemented 2>/dev/null || true
# ffmpeg-python is what webui.py actually needs for `import ffmpeg`
!pip install -q ffmpeg-python
print('[+] Requirements installed')

# ── 6c: Copy our filelist ─────────────────────────────────────────────────────
print('[3/4] Copying Hindi filelist...')
!cp /content/hindi_tts/training_data/sovits_filelist.txt /content/GPT-SoVITS/hindi_filelist.txt
lines = open('/content/GPT-SoVITS/hindi_filelist.txt').readlines()
print(f'[+] Filelist ready: {len(lines)} segments')
print(f'    Sample: {lines[0][:100] if lines else "(empty)"}')

# ── 6d: Verify ffmpeg import ──────────────────────────────────────────────────
print('[4/4] Verifying ffmpeg import...')
import ffmpeg
print('[+] import ffmpeg OK')
print()
print('🚀 Launching GPT-SoVITS WebUI...')
print('   Training filelist path → /content/GPT-SoVITS/hindi_filelist.txt')
print()

# Launch WebUI with public Gradio share link
!python webui.py --share

---
## 💾 Save Outputs to Google Drive (Run After Training)

In [ ]:
import shutil, os
from google.colab import drive

# Make sure Drive is mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

SAVE_TO = '/content/drive/MyDrive/hindi_tts_outputs'
os.makedirs(SAVE_TO, exist_ok=True)

# Save metadata + filelist
shutil.copy('/content/hindi_tts/metadata.csv',
            os.path.join(SAVE_TO, 'metadata.csv'))
shutil.copy('/content/hindi_tts/training_data/sovits_filelist.txt',
            os.path.join(SAVE_TO, 'sovits_filelist.txt'))

# Save GPT-SoVITS model checkpoints if they exist
CKPT_DIRS = [
    '/content/GPT-SoVITS/logs',
    '/content/GPT-SoVITS/SoVITS_weights',
    '/content/GPT-SoVITS/GPT_weights',
]
for d in CKPT_DIRS:
    if os.path.exists(d):
        dest = os.path.join(SAVE_TO, os.path.basename(d))
        shutil.copytree(d, dest, dirs_exist_ok=True)
        print(f'[+] Saved {d} → {dest}')

print(f'\n✅ All outputs saved to Google Drive: {SAVE_TO}')

---
## 🎧 Bonus: Instant Hindi TTS Test (Edge-TTS — No Training Needed)

In [ ]:
import asyncio, os
import edge_tts
from IPython.display import Audio, display

TEXT   = 'नमस्ते! यह हिंदी टेक्स्ट-टू-स्पीच का परीक्षण है। आज का दिन बहुत अच्छा है।'
VOICE  = 'hi-IN-MadhurNeural'   # male narrator
OUTPUT = '/content/hindi_tts/test_tts.mp3'

async def gen():
    comm = edge_tts.Communicate(text=TEXT, voice=VOICE, rate='+0%', pitch='+0Hz')
    await comm.save(OUTPUT)

await gen()
print(f'[+] Audio saved: {OUTPUT}')
display(Audio(OUTPUT, autoplay=True))